# 09 · Event-time windows, late events, and replayable state

**Level:** advanced · **Time:** 50–65 minutes · **Prerequisites:** notebooks 02–06.

Build a small Python stream processor that sums purchases in ten-second tumbling
windows. Observe out-of-order arrival, a watermark, an event that arrives too late,
final results, and rebuilding state by replaying the source.

This is an explicit Python teaching implementation, not the Java Kafka Streams API.
It uses one source partition so the event sequence is deterministic. Start Kafka,
restart the kernel, and run top to bottom.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
from collections import defaultdict

require_broker()
source_topic = new_topic("09.purchases", partitions=1)
totals_topic = new_topic("09.window-totals", partitions=1,
                         config={"cleanup.policy": "compact"})
late_topic = new_topic("09.late", partitions=1)


## 1. Arrival order and event time are different

`event_second` is a timestamp measured from the beginning of our fictional session.
Windows are `[0,10)`, `[10,20)`, and so on: the right endpoint is excluded.
Kafka preserves order within this partition, but producers can send event timestamps
out of order. Offsets tell us arrival order, not when the business event happened.


In [ ]:
purchases = [
    {"event_id": "e1", "customer_id": "c1", "event_second": 2, "amount_paise": 10},
    {"event_id": "e2", "customer_id": "c1", "event_second": 7, "amount_paise": 20},
    {"event_id": "e3", "customer_id": "c1", "event_second": 12, "amount_paise": 30},
    {"event_id": "e4", "customer_id": "c2", "event_second": 6, "amount_paise": 7},
    {"event_id": "e5", "customer_id": "c1", "event_second": 25, "amount_paise": 40},
    {"event_id": "e6", "customer_id": "c1", "event_second": 1, "amount_paise": 999},
    {"event_id": "e7", "customer_id": "c2", "event_second": 36, "amount_paise": 50},
]
publish(source_topic, purchases, key_field="customer_id")
print([(x["event_id"], x["event_second"]) for x in purchases])


## 2. Define the watermark policy before writing code

Our watermark is `maximum event time observed − 5 seconds`. Finalize a window when
its end is less than or equal to that watermark. A new event for an already finalized
window is late and goes to a side topic. This is our policy, not a guarantee that all
earlier events really arrived. A future-dated event can advance the watermark too far.

Event `e4` is out of order but its window remains open. Event `e6` arrives after the
watermark has closed its window. Predict which one will contribute to the result.


In [ ]:
WINDOW_SECONDS = 10
ALLOWED_LATENESS = 5

def window_start(second):
    return (second // WINDOW_SECONDS) * WINDOW_SECONDS

assert [window_start(t) for t in [0, 9, 10, 19, 20]] == [0, 0, 10, 10, 20]


## 3. Implement the state machine

Each open `(customer, window)` key holds a sum and count. Finalization emits a stable
key and removes that state. A hard cap prevents accidental unbounded key cardinality;
a real processor needs a suitable persistent state store, backpressure, and monitoring.


In [ ]:
class WindowTotals:
    def __init__(self, max_open_keys=1000):
        self.max_seen = None
        self.watermark = float("-inf")
        self.state = {}
        self.max_open_keys = max_open_keys

    def emit_closed(self, watermark):
        results = []
        ready = [k for k in self.state if k[1] + WINDOW_SECONDS <= watermark]
        for customer, start in sorted(ready):
            total, count = self.state.pop((customer, start))
            results.append({"window_id": f"{customer}:{start}",
                            "customer_id": customer, "window_start": start,
                            "window_end": start + WINDOW_SECONDS,
                            "total_paise": total, "event_count": count})
        return results

    def process(self, event):
        second = event["event_second"]
        if type(second) is not int or second < 0:
            raise ValueError("event_second must be a nonnegative integer")
        if type(event["amount_paise"]) is not int or event["amount_paise"] <= 0:
            raise ValueError("amount_paise must be a positive integer")
        start = window_start(second)
        self.max_seen = second if self.max_seen is None else max(self.max_seen, second)
        self.watermark = self.max_seen - ALLOWED_LATENESS
        closed = self.emit_closed(self.watermark)
        if start + WINDOW_SECONDS <= self.watermark:
            late = {**event, "watermark": self.watermark,
                    "reason": "window_already_finalized"}
            return closed, late
        key = (event["customer_id"], start)
        if key not in self.state and len(self.state) >= self.max_open_keys:
            raise RuntimeError("Open-window state limit reached; stop and investigate")
        total, count = self.state.get(key, (0, 0))
        self.state[key] = (total + event["amount_paise"], count + 1)
        return closed, None

    def finish_finite_sample(self):
        # Legal only because our finite input batch has an explicit end.
        return self.emit_closed(float("inf"))


## 4. Inspect the algorithm before connecting it to the consumer

The trace shows each watermark, the remaining open keys, and finalized output.
An idle live stream has no explicit end: a silent source does not automatically advance
this event-time watermark. Do not call the finite-sample finish operation on a live stream.


In [ ]:
preview = WindowTotals()
preview_results, preview_late = [], []
for event in purchases:
    closed, late = preview.process(event)
    preview_results.extend(closed)
    if late:
        preview_late.append(late)
    print(event["event_id"], "watermark=", preview.watermark,
          "open_keys=", len(preview.state), "closed=", len(closed),
          "late=", late is not None)
preview_results.extend(preview.finish_finite_sample())
assert [x["event_id"] for x in preview_late] == ["e6"]
assert preview.state == {}


## 5. Run the processor against actual Kafka records

For this lab we deliberately leave source offsets uncommitted and rebuild from the
complete retained input on restart. Committing an offset while all window state exists
only in memory would lose that state after a crash. This full-replay design has increasing
recovery cost and needs source retention long enough for a complete reconstruction.

Each final result uses `window_id` as its Kafka key. Consumers should overwrite the
previous result for that key, rather than add the same total again. Compaction is
asynchronous; duplicate versions can exist in the log until cleaning occurs.


In [ ]:
rebuild_group = f"lab-09-rebuild-{uuid4().hex}"

def run_finite_job(destination, late_destination):
    processor = WindowTotals()
    reader = consumer(rebuild_group)
    emitted, late_count = [], 0
    try:
        reader.subscribe([source_topic])
        for _ in range(len(purchases)):
            message = poll_records(reader, count=1, timeout=20)[0]
            closed, late = processor.process(decode(message))
            if closed:
                publish(destination, closed, key_field="window_id")
                emitted.extend(closed)
            if late:
                publish(late_destination, [late], key_field="event_id")
                late_count += 1
        final = processor.finish_finite_sample()
        publish(destination, final, key_field="window_id")
        emitted.extend(final)
        # Intentionally no source commit: there is no durable state checkpoint.
    finally:
        reader.close()
    return emitted, late_count

emitted, late_count = run_finite_job(totals_topic, late_topic)
print("Finalized windows:", len(emitted), "late events:", late_count)


In [ ]:
totals = [decode(m) for m in collect(totals_topic, count=5)]
late_events = [decode(m) for m in collect(late_topic, count=1)]
actual = {(x["customer_id"], x["window_start"]): x["total_paise"] for x in totals}
expected = {("c1", 0): 30, ("c2", 0): 7, ("c1", 10): 30,
            ("c1", 20): 40, ("c2", 30): 50}
assert actual == expected
assert late_events[0]["event_id"] == "e6"
assert sum(x["event_count"] for x in totals) == 6
print(actual)


**Observe:** six purchases contributed to five customer windows. The value 999 never
enters a final total because its window was already finalized. A different lateness
policy can produce different results from the same arrival sequence.


## 6. Rebuild from the same group and compare the results

The group has no committed offset, so `earliest` selects the beginning again. We use
fresh output topics to make the two executions easy to compare. A real rebuild could
write versioned output and atomically switch readers after verification.


In [ ]:
rebuilt_topic = new_topic("09.rebuilt-totals", partitions=1,
                          config={"cleanup.policy": "compact"})
rebuilt_late_topic = new_topic("09.rebuilt-late", partitions=1)
rebuilt, rebuilt_late_count = run_finite_job(rebuilt_topic, rebuilt_late_topic)
observed_rebuild = [decode(m) for m in collect(rebuilt_topic, count=5)]
by_id = lambda values: {x["window_id"]: x for x in values}
assert by_id(observed_rebuild) == by_id(totals)
assert rebuilt_late_count == late_count == 1
print("Replayed input reconstructed identical window results.")


## 7. From this demonstration to a durable stream processor

Incremental recovery requires coordinating source offsets, window state, and output.
A checkpoint file written separately from an offset commit leaves a crash gap. An
output transaction alone does not preserve arbitrary in-memory Python state. Possible
designs include a transactional database holding state and source positions, or a
streaming runtime with a changelog/state-store recovery model.

With multiple input partitions, using a single global maximum timestamp can close
windows prematurely when another partition lags. Real watermarks usually account for
per-partition progress and an explicit idle-partition policy. Validate future timestamps,
handle duplicate event IDs, and budget state cardinality before deploying.


## Practice

1. Raise allowed lateness to 30 seconds and rerun the preview. Is `e6` still late?
2. Move `e6` before `e5` and explain the changed result.
3. Duplicate `e1`. Does this processor deduplicate it?
4. Create more than 1,000 distinct open customer windows and observe the state cap.

**Solution hints:** the first window stays open longer; arrival order affects finalization;
this processor counts duplicate input again; the cap intentionally fails instead of
silently using unlimited memory. A production deduplication store also needs retention.


## Presenter checkpoint and references

Ask the learner to explain event time, arrival order, a watermark, finalization, and why
committing offsets without durable window state is unsafe.

[Kafka Streams core concepts](https://kafka.apache.org/41/streams/core-concepts/)
provides terminology; [Kafka log compaction](https://kafka.apache.org/41/design/design/#log-compaction)
explains keyed retention. The code above is our Python implementation of the stated policy.
